# Introduction

## Satellite Tracking & Orbital Analysis — ISS

This project analyzes 90 days of historical Two-Line Element (TLE) data for the 
International Space Station (ISS) retrieved from US Space Command's space-track.org.

**Central Question:** How does the ISS orbital altitude change over time due to 
atmospheric drag and reboost maneuvers, and can we forecast its future altitude decay?

Using the sgp4 orbital mechanics library we compute the ISS ground track and 
visualize its orbital trajectory on an interactive 3D globe. A Prophet time series 
model is then used to forecast altitude trends over the next 30 days.

In [6]:
%pip install spacetrack sgp4 folium plotly prophet
# Do a check to see if the packages were installed correctly

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Standard libraries
import os
import io
import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import folium
import branca.colormap as cm
from folium.plugins import AntPath

# Satellite tracking
from spacetrack import SpaceTrackClient
import spacetrack.operators as op
from sgp4.api import Satrec, jday

# Forecasting
from prophet import Prophet

# Global color palette
COLORS = {
    "primary": "#2196F3",      # blue
    "secondary": "#FF5722",    # orange/red  
    "accent": "#4CAF50",       # green
    "neutral": "#9E9E9E",      # gray
    "background": "#1E1E1E",   # dark background
    "text": "#FFFFFF"          # white text
}

# Global plot style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12
})

## Data Collection

In [ ]:
# Connect to space-track.org
st = SpaceTrackClient(
    identity=os.environ.get('SPACETRACK_EMAIL'),
    password=os.environ.get('SPACETRACK_PASSWORD')
)

# Pull ISS TLE histroy for the last 90 days
end = datetime.utcnow()
start = end - timedelta(days=90)
drange = op.inclusive_range(start, end)

data = st.gp_history(
    norad_cat_id = 25544, # ISS NORAD ID
    orderby = 'epoch asc',
    format = 'csv',
    epoch = drange
)

df = pd.read_csv(io.StringIO(data))
print(df.shape)
print(df.columns.tolist())
df.head()


/var/folders/hh/7l31dxnn5fb61khx6fl_ytch0000gn/T/ipykernel_69117/1896996395.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


(378, 40)
['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'OBJECT_NAME', 'OBJECT_ID', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS', 'OBJECT_TYPE', 'RCS_SIZE', 'COUNTRY_CODE', 'LAUNCH_DATE', 'SITE', 'DECAY_DATE', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2']


,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,...,RCS_SIZE,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-01-22T10:08:43,18 SPCS,ISS (ZARYA),1998-067A,EARTH,TEME,UTC,SGP4,...,LARGE,CIS,1998-11-20,TTMTR,NaN,4974330,309613550,0 ISS (ZARYA),1 25544U 98067A 26022.17441205 .00016198 0...,2 25544 51.6328 302.3013 0007809 45.1395 315...
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-01-22T14:46:49,18 SPCS,ISS (ZARYA),1998-067A,EARTH,TEME,UTC,SGP4,...,LARGE,CIS,1998-11-20,TTMTR,NaN,4974861,309653296,0 ISS (ZARYA),1 25544U 98067A 26022.23890398 .00015740 0...,2 25544 51.6328 301.9816 0007710 45.6250 314...
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-01-23T04:38:27,18 SPCS,ISS (ZARYA),1998-067A,EARTH,TEME,UTC,SGP4,...,LARGE,CIS,1998-11-20,TTMTR,NaN,4975303,309666379,0 ISS (ZARYA),1 25544U 98067A 26022.94831487 .00013618 0...,2 25544 51.6328 298.4677 0007749 48.1765 311...
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-01-23T21:30:39,18 SPCS,ISS (ZARYA),1998-067A,EARTH,TEME,UTC,SGP4,...,LARGE,CIS,1998-11-20,TTMTR,NaN,4975627,309699308,0 ISS (ZARYA),1 25544U 98067A 26023.39975333 .00015276 0...,2 25544 51.6326 296.2323 0007748 49.8407 310...
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-01-24T04:06:48,18 SPCS,ISS (ZARYA),1998-067A,EARTH,TEME,UTC,SGP4,...,LARGE,CIS,1998-11-20,TTMTR,NaN,4975749,309731142,0 ISS (ZARYA),1 25544U 98067A 26023.72220588 .00016677 0...,2 25544 51.6325 294.6353 0007776 51.0046 309...


In [9]:
# Select relevant columns

keep = ["EPOCH", "MEAN_MOTION", "ECCENTRICITY", "INCLINATION", 
        "APOAPSIS", "PERIAPSIS", "SEMIMAJOR_AXIS", "PERIOD", 
        "BSTAR", "TLE_LINE1", "TLE_LINE2"]

df = df[keep]
df["EPOCH"] = pd.to_datetime(df["EPOCH"])
print(df.shape)
df.head()

(378, 11)


,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,APOAPSIS,PERIAPSIS,SEMIMAJOR_AXIS,PERIOD,BSTAR,TLE_LINE1,TLE_LINE2
0,2026-01-22 04:11:09.201120,15.495481,0.000781,51.6328,423.356,412.742,6796.184,92.930,0.000296,1 25544U 98067A 26022.17441205 .00016198 0...,2 25544 51.6328 302.3013 0007809 45.1395 315...
1,2026-01-22 05:44:01.303872,15.495502,0.000771,51.6328,423.283,412.803,6796.178,92.930,0.000287,1 25544U 98067A 26022.23890398 .00015740 0...,2 25544 51.6328 301.9816 0007710 45.6250 314...
2,2026-01-22 22:45:34.404768,15.495692,0.000775,51.6328,423.254,412.721,6796.122,92.929,0.000250,1 25544U 98067A 26022.94831487 .00013618 0...,2 25544 51.6328 298.4677 0007749 48.1765 311...
3,2026-01-23 09:35:38.687712,15.495836,0.000775,51.6326,423.211,412.680,6796.080,92.928,0.000279,1 25544U 98067A 26023.39975333 .00015276 0...,2 25544 51.6326 296.2323 0007748 49.8407 310...
4,2026-01-23 17:19:58.588032,15.495963,0.000778,51.6325,423.193,412.623,6796.043,92.927,0.000304,1 25544U 98067A 26023.72220588 .00016677 0...,2 25544 51.6325 294.6353 0007776 51.0046 309...


## Data Collection Summary
- Connected to space-track.org API using SpaceTrackClient
- Retrieved 364 TLE records for the ISS (NORAD ID: 25544)
- Date range: 90 days of historical orbital data
- Retained 11 key orbital parameters for analysis
- Converted EPOCH to datetime format

## Exploratory Analysis

In [34]:
# Altitude plot

fig = go.Figure()

# Apoapsis line
fig.add_trace(go.Scatter(
    x=df["EPOCH"],
    y=df["APOAPSIS"],
    mode="lines",
    line=dict(color=COLORS["primary"], width=2),
    name="Apoapsis"
))

# Periapsis line
fig.add_trace(go.Scatter(
    x=df["EPOCH"],
    y=df["PERIAPSIS"],
    mode="lines",
    line=dict(color=COLORS["secondary"], width=2),
    name="Periapsis"
))

# Reboost annotation
max_jump_idx = df["APOAPSIS"].diff().idxmax()
reboost_date = df.loc[max_jump_idx, "EPOCH"]
reboost_alt = df.loc[max_jump_idx, "APOAPSIS"]

fig.add_annotation(
    x=reboost_date,
    y=reboost_alt,
    text="Reboost Event",
    showarrow=True,
    arrowhead=2,
    arrowcolor="white",
    font=dict(color="white", size=11),
    bgcolor=COLORS["accent"],
    bordercolor=COLORS["accent"],
    ay=-40
)

# Stats text box
mean_alt = df["APOAPSIS"].mean()
decay_rate = (df["APOAPSIS"].iloc[-1] - df["APOAPSIS"].iloc[0]) / len(df)

fig.add_annotation(
    x=0.5,
    y=0.5,
    xref="paper",
    yref="paper",
    text=f"Mean Apoapsis: {mean_alt:.1f} km | Decay Rate: {decay_rate:.3f} km/update",
    showarrow=False,
    font=dict(color="white", size=10),
    bgcolor=COLORS["primary"],
    opacity=0.7
)

fig.update_layout(
    title="ISS Orbital Altitude Over Time",
    xaxis_title="Date",
    yaxis_title="Altitude (km)",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#1E1E1E",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(0,0,0,0.5)")
)

fig.show()

**Figure 1:** The ISS orbital altitude is not stable — it reflects an ongoing battle between 
atmospheric drag pulling the station down and periodic reboost maneuvers pushing 
it back up. The sharp apoapsis spike indicates a major reboost event, while the 
staircase pattern in periapsis reflects multiple smaller thruster burns. Without 
regular reboosts, the ISS would naturally decay and reenter Earth's atmosphere 
within a few years.

## Exploratory Analysis

In [11]:
# Position calculation using SGP4

positions = []

for _, row in df.iterrows():
    satellite = Satrec.twoline2rv(row["TLE_LINE1"], row["TLE_LINE2"])
    
    # Convert epoch to julian date
    epoch = row["EPOCH"]
    jd, fr = jday(epoch.year, epoch.month, epoch.day, 
                  epoch.hour, epoch.minute, epoch.second)
    
    # Get position in km (x, y, z in Earth-centered inertial frame)
    e, r, v = satellite.sgp4(jd, fr)
    
    if e == 0:  # 0 means no error
        positions.append({
            "epoch": epoch,
            "x": r[0], "y": r[1], "z": r[2]  # position in km
        })

pos_df = pd.DataFrame(positions)
print(pos_df.shape)
pos_df.head()

(378, 4)


,epoch,x,y,z
0,2026-01-22 04:11:09.201120,3629.272416,-5742.438695,-1.206288
1,2026-01-22 05:44:01.303872,3596.797482,-5762.922367,-1.826251
2,2026-01-22 22:45:34.404768,3236.444910,-5972.853310,-2.435131
3,2026-01-23 09:35:38.687712,2999.837024,-6095.208260,-4.134719
4,2026-01-23 17:19:58.588032,2829.247049,-6176.276046,-3.535524


## Data Processing And Orbital Position Calculation

In [12]:
# Timestamp conversion and geodetic coordinates

def eci_to_latlon(x, y, z):
    # Distance from Earth's center
    r = np.sqrt(x**2 + y**2 + z**2)
    
    # Latitude
    lat = np.degrees(np.arcsin(z / r))
    
    # Longitude
    lon = np.degrees(np.arctan2(y, x))
    
    return lat, lon

pos_df["lat"] = pos_df.apply(lambda row: eci_to_latlon(row["x"], row["y"], row["z"])[0], axis=1)
pos_df["lon"] = pos_df.apply(lambda row: eci_to_latlon(row["x"], row["y"], row["z"])[1], axis=1)
pos_df["altitude"] = np.sqrt(pos_df["x"]**2 + pos_df["y"]**2 + pos_df["z"]**2) - 6371  # subtract Earth's radius

print(pos_df.head())

                       epoch            x            y         z       lat  \
0 2026-01-22 04:11:09.201120  3629.272416 -5742.438695 -1.206288 -0.010174   
1 2026-01-22 05:44:01.303872  3596.797482 -5762.922367 -1.826251 -0.015403   
2 2026-01-22 22:45:34.404768  3236.444910 -5972.853310 -2.435131 -0.020538   
3 2026-01-23 09:35:38.687712  2999.837024 -6095.208260 -4.134719 -0.034872   
4 2026-01-23 17:19:58.588032  2829.247049 -6176.276046 -3.535524 -0.029818   

         lon    altitude  
0 -57.706749  422.174655  
1 -58.030585  422.248830  
2 -61.548548  422.346616  
3 -63.795287  422.423511  
4 -65.388289  422.455465  


In [13]:
# Visualize the trajectory on a map

print(pos_df.columns.tolist())
print(pos_df[["lat", "lon", "altitude"]].head())

['epoch', 'x', 'y', 'z', 'lat', 'lon', 'altitude']
        lat        lon    altitude
0 -0.010174 -57.706749  422.174655
1 -0.015403 -58.030585  422.248830
2 -0.020538 -61.548548  422.346616
3 -0.034872 -63.795287  422.423511
4 -0.029818 -65.388289  422.455465


In [14]:
# Create a folium map centered around the average position


# Generate timestamps every 5 minutes
start_time = pos_df["epoch"].min()
end_time = pos_df["epoch"].max()

timestamps = []
current = start_time
while current <= end_time:
    timestamps.append(current)
    current += timedelta(minutes=5)

print(f"Total timestamps: {len(timestamps):,}")

Total timestamps: 25,723


In [15]:
# Use the most recent TLE (two-line elements)for position calculations

satellite = Satrec.twoline2rv(df["TLE_LINE1"].iloc[-1], df["TLE_LINE2"].iloc[-1])

track = []

for t in timestamps:
    jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute, t.second)
    e, r, v = satellite.sgp4(jd, fr)
    
    if e == 0:
        lat, lon = eci_to_latlon(r[0], r[1], r[2])
        alt = np.sqrt(r[0]**2 + r[1]**2 + r[2]**2) - 6371
        track.append({"time": t, "lat": lat, "lon": lon, "altitude": alt})

track_df = pd.DataFrame(track)
print(track_df.shape)
track_df.head()

(25723, 4)


,time,lat,lon,altitude
0,2026-01-22 04:11:09.201120,11.855114,112.563561,435.564918
1,2026-01-22 04:16:09.201120,-3.242349,124.691615,437.986610
2,2026-01-22 04:21:09.201120,-18.181656,137.183655,439.393814
3,2026-01-22 04:26:09.201120,-32.158729,151.959430,439.727911
4,2026-01-22 04:31:09.201120,-43.834300,171.588273,439.147053


In [16]:
# Create a 2-d interactive Plotly map of the ISS ground track

recent = track_df.tail(90)

m = folium.Map(location=[0, 0], zoom_start=2, tiles="CartoDB dark_matter")

# Create color map based on altitude
colormap = cm.linear.RdYlGn_09.scale(recent["altitude"].min(), recent["altitude"].max())
colormap.caption = "ISS Altitude (km)"
colormap.add_to(m)

def split_antimeridian(df):
    segments = []
    current_segment = [df.iloc[0]]
    
    for i in range(1, len(df)):
        if abs(df.iloc[i]["lon"] - df.iloc[i-1]["lon"]) > 180:
            segments.append(pd.DataFrame(current_segment))
            current_segment = []
        current_segment.append(df.iloc[i])
    
    segments.append(pd.DataFrame(current_segment))
    return segments

segments = split_antimeridian(recent)

# Draw track colored by altitude
for segment in segments:
    for i in range(len(segment) - 1):
        p1 = segment.iloc[i]
        p2 = segment.iloc[i + 1]
        color = colormap(p1["altitude"])
        folium.PolyLine(
            [[p1["lat"], p1["lon"]], [p2["lat"], p2["lon"]]],
            color=color,
            weight=2.5,
            opacity=0.9
        ).add_to(m)

# Current ISS position
latest = track_df.iloc[-1]
folium.Marker(
    location=[latest["lat"], latest["lon"]],
    popup=f"ISS Current Position\nAltitude: {latest['altitude']:.1f} km\nLat: {latest['lat']:.2f}°\nLon: {latest['lon']:.2f}°",
    icon=folium.Icon(color="red", icon="star")
).add_to(m)

# Add ground stations
ground_stations = [
    {"name": "NASA Johnson Space Center", "lat": 29.5590, "lon": -95.0930},
    {"name": "NASA Goddard Space Center", "lat": 38.9963, "lon": -76.8483},
    {"name": "Roscosmos Moscow", "lat": 55.7558, "lon": 37.6173},
    {"name": "JAXA Tsukuba", "lat": 36.0565, "lon": 140.1254},
    {"name": "ESA Darmstadt", "lat": 49.8713, "lon": 8.6221},
]

for station in ground_stations:
    folium.CircleMarker(
        location=[station["lat"], station["lon"]],
        radius=6,
        color="white",
        fill=True,
        fill_color="white",
        fill_opacity=0.8,
        popup=station["name"]
    ).add_to(m)

m.save("iss_ground_track.html")
print("Map saved!")

Map saved!


In [17]:
# Create a 3-d interactive Plotly map of the ISS ground track

sample = track_df.tail(90).iloc[::2].reset_index(drop=True)

fig = go.Figure()

# Color coded track by altitude
fig.add_trace(go.Scattergeo(
    lat=sample["lat"],
    lon=sample["lon"],
    mode="lines+markers",
    marker=dict(
        size=3,
        color=sample["altitude"],
        colorscale="RdYlGn",
        showscale=True,
        colorbar=dict(title="Altitude (km)", x=1.02)
    ),
    line=dict(width=1.5, color="lime"),
    name="ISS Track"
))

# Current position
fig.add_trace(go.Scattergeo(
    lat=[latest["lat"]],
    lon=[latest["lon"]],
    mode="markers+text",
    marker=dict(size=12, color="red", symbol="star"),
    text=["🛸 ISS"],
    textposition="top right",
    textfont=dict(size=12, color="white"),
    name=f"Current Position ({latest['altitude']:.1f} km)"
))

# Ground stations
for station in ground_stations:
    fig.add_trace(go.Scattergeo(
        lat=[station["lat"]],
        lon=[station["lon"]],
        mode="markers+text",
        marker=dict(size=6, color="white"),
        text=station["name"],
        textposition="top right",
        textfont=dict(size=8, color="white"),
        showlegend=False
    ))

fig.update_layout(
    title="ISS Ground Track — Last 5 Orbits",
    geo=dict(
        projection_type="orthographic",
        showland=True,
        landcolor="darkgray",
        showocean=True,
        oceancolor="midnightblue",
        showcoastlines=True,
        coastlinecolor="white",
        coastlinewidth=0.5,
        bgcolor="black"
    ),
    paper_bgcolor="black",
    font=dict(color="white"),
    legend=dict(
        bgcolor="rgba(0,0,0,0.5)",
        x=-0.65,        # move legend to left side
        y=1.10,
        xanchor="left",
        yanchor="top"
    ),
    coloraxis_colorbar=dict(
        x=1.05        # push colorbar further right
    )
)

fig.show()

## Forecasting

In [35]:
# Forecasting periapsis altitude with Prophet

# Prepare data for Prophet
prophet_df = df[["EPOCH", "PERIAPSIS"]].rename(columns={"EPOCH": "ds", "PERIAPSIS": "y"})

model = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
model.fit(prophet_df)

# Forecast 30 days ahead
future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

# Plot actual data

# Calculate 95% interval manually
margin = (forecast["yhat_upper"] - forecast["yhat_lower"]) * 0.9

fig = go.Figure()

# 95% confidence band
fig.add_trace(go.Scatter(
    x=pd.concat([forecast["ds"], forecast["ds"][::-1]]),
    y=pd.concat([forecast["yhat_upper"] + margin, (forecast["yhat_lower"] - margin)[::-1]]),
    fill="toself",
    fillcolor="rgba(255, 87, 34, 0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    name="95% Confidence"
))

# 80% confidence band
fig.add_trace(go.Scatter(
    x=pd.concat([forecast["ds"], forecast["ds"][::-1]]),
    y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"][::-1]]),
    fill="toself",
    fillcolor="rgba(255, 87, 34, 0.3)",
    line=dict(color="rgba(255,255,255,0)"),
    name="80% Confidence"
))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast["ds"],
    y=forecast["yhat"],
    mode="lines",
    line=dict(color="#FF5722", width=2),
    name="Forecast"
))

# Actual data points
fig.add_trace(go.Scatter(
    x=prophet_df["ds"],
    y=prophet_df["y"],
    mode="markers",
    marker=dict(size=5, color="#2196F3"),
    name="Actual"
))

# Forecast start line
forecast_start = prophet_df["ds"].max()
y_min = forecast["yhat"].min()
y_max = forecast["yhat"].max()

fig.add_trace(go.Scatter(
    x=[forecast_start, forecast_start],
    y=[y_min, y_max],
    mode="lines",
    line=dict(color="gray", dash="dash", width=1.5),
    name="Forecast Start"
))

fig.update_layout(
    title="ISS Periapsis Altitude Forecast with Uncertainty Bands",
    xaxis_title="Date",
    yaxis_title="Altitude (km)",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#1E1E1E",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(0,0,0,0.5)")
)

# Calculate and display decay rate
forecast_period = forecast[forecast["ds"] > prophet_df["ds"].max()]
daily_decay = (forecast_period["yhat"].iloc[-1] - forecast_period["yhat"].iloc[0]) / 30

fig.add_annotation(
    x=0.98,
    y=0.05,
    xref="paper",
    yref="paper",
    text=f"Predicted Decay Rate: {daily_decay:.3f} km/day",
    showarrow=False,
    font=dict(color="white", size=10),
    bgcolor=COLORS["secondary"],
    opacity=0.8,
    xanchor="right"
)

fig.show()

18:11:57 - cmdstanpy - INFO - Chain [1] start processing
18:11:57 - cmdstanpy - INFO - Chain [1] done processing


In [29]:
print(type(prophet_df["ds"].max()))
print(prophet_df["ds"].max())

<class 'pandas._libs.tslibs.timestamps.Timestamp'>
2026-04-21 11:45:46.937376


The forecast shows a continued gradual decline in ISS periapsis altitude over the 
next 30 days driven by atmospheric drag. The 80% confidence interval (darker band) captures the most likely range of outcomes, while the 95% interval (lighter band) reflects broader uncertainty. The widening bands over time reflect a fundamental limitation of time series forecasting — the further into the future we predict, the less certain we can be. Notably the model cannot anticipate reboost maneuvers, which would cause sudden upward jumps outside any predicted interval.

## Key Findings

- **Atmospheric drag causes a measurable and continuous altitude decline** of approximately 
  0.03 km/day, forcing periodic reboost maneuvers by Mission Control to maintain the ISS's 
  operational orbit. The reboost events are clearly visible as sharp altitude spikes in the 
  historical TLE data.

- **SGP4 orbital mechanics confirmed the ISS maintains a nearly circular orbit** (eccentricity ≈ 0) 
  at ~420km altitude. Its 51.6° inclination provides ground track coverage over approximately 
  90% of Earth's populated landmass, a deliberate design choice reflecting the international 
  nature of the station.

- **Prophet time series forecasting predicts continued altitude decay** over the next 30 days, 
  consistent with the historical drag trend. Uncertainty intervals widen significantly beyond 
  2 weeks, reflecting the model's inability to anticipate future reboost maneuvers which would 
  cause sudden upward altitude jumps outside any predicted range.

- **The 3D interactive ground track visualization** demonstrates the characteristic sinusoidal 
  orbital pattern — each successive pass shifts approximately 22.5° westward as Earth rotates 
  beneath the station, with the ISS completing roughly 15.5 orbits per day.

## Conclusion

This study set out to analyze ISS orbital patterns using real Two-Line Element data from US Space Command and forecast future altitude trends using time series modeling.

By combining TLE data from space-track.org with SGP4 orbital mechanics calculations we were able to compute real satellite positions, visualize the ISS ground track on an interactive 3D globe, and quantify the effects of atmospheric drag on orbital altitude. The data revealed a consistent decay rate of ~0.03 km/day, punctuated by periodic reboost maneuvers that temporarily restore altitude.

Prophet forecasting captured the underlying decay trend well, though its primary 
limitation is the inability to anticipate scheduled reboosts — discrete human 
interventions that fall outside the scope of any purely data-driven model.

Future work could extend this analysis to track multiple satellites simultaneously and compare decay rates across different orbital altitudes. Incorporating solar activity data would also improve forecast accuracy, as solar flares increase atmospheric density and accelerate drag. Most ambitiously, applying anomaly detection to orbital changes could help identify suspicious maneuvers or potential collision risks — a real application in space domain awareness.